## Data Ingestion Patterns

### 1. Imports

In [ ]:
import requests
from pathlib import Path

In [ ]:
DATASET_URL = "https://huggingface.co/datasets/allenai/c4/resolve/main"
OUT_DIR = Path("data/c4/en")

In [ ]:
def shard_url(i: int, split: str ="train", total: int = 1024):
      return f"{DATASET_URL}/en/c4-{split}.{i:05d}-of-{total:05d}.json.gz"

In [ ]:
def fetch(url: str, out_dir: Path, chunk_size: int = 2**20):
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    part = out_dir.with_name(out_dir.name + ".part")
    have = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={have}-"} if have else {}

    with requests.get(url, headers=headers, stream=True, timeout=(10, 120)) as r:
        r.raise_for_status()
        if have and r.status_code != 206:
            have = 0
            raise RuntimeError(f"Server does not support resuming downloads: {url}")
        want = int(r.headers["Content-Length"]) + have
        with open(part, "ab" if have else "wb") as f:
            for block in r.iter_content(chunk_size=chunk_size):
                f.write(block)

        got = part.stat().st_size
        if got != want:
            raise RuntimeError(f"Download incomplete: {got} != {want}")
        part.rename(out_dir)

In [ ]:
for i in range(12):
    dest = OUT_DIR / f"c4-train.{i:05d}-of-1024.json.gz"
    if not dest.exists():
        fetch(shard_url(i), dest)